## Instalando bibliotecas

In [1]:
!pip install transformers torch accelerate pdfplumber tqdm -q

## Importando bibliotecas

In [1]:
import pdfplumber
import json
import torch
from transformers import pipeline, logging
from tqdm import tqdm

# Desativa avisos (apenas erros críticos serão mostrados)
logging.set_verbosity_error()

/home/jessica/Área de trabalho/topicosAvancadosEmIAAAvaliacao02/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Função para extrair texto do pdf usando o pdfplumber

In [2]:
def extract_text_pdf(file_path):

    if file_path.lower().endswith('manual_genie.pdf'):
        text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text

## Divisão em trechos (chunks)

In [3]:
def split_text(text, max_chunk_length=600):

    paragraphs = text.split("\n")
    chunks = []
    current_chunk = ""

    for para in paragraphs:
        if len(current_chunk) + len(para) < max_chunk_length:
            current_chunk += para + "\n"
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = para + "\n"

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

## Gerando perguntas e respostas

In [4]:
def generate_instruction_response(chunk, hf_pipeline):

    prompt = f"""
                Você é um gerador de dados para o ajuste fino ou treinamento de um modelo de linguagem.

                #Dado o conteúdo e diretrizes abaixo, crie:
                - Priorize especificações técnicas e normas de segurança encontradas no texto.
                - Baseie-se ESTRITAMENTE no conteúdo fornecido; não utilize conhecimento externo nem invente informações.
                - Não adicione explicações extras ou introduções.
                - Uma pergunta (instrução) curta e objetiva, com menos de 25 palavras, baseada apenas no conteúdo.
                - Uma resposta direta e concisa (resposta), com menos de 25 palavras, baseada apenas no conteúdo.

                Use o seguinte formato:
                INSTRUCTION: <pergunta>
                RESPONSE: <resposta>

                CONTEÚDO:
                \"\"\"
                {chunk}
                \"\"\"
                """

    messages = [{"role": "user", "content": prompt}]

    try:
        outputs = hf_pipeline(
            messages,
            max_new_tokens=150,
            max_length=None,  
            return_full_text=False
        )
        content = outputs[0]["generated_text"]

        # Extrai os campos esperados
        instr_part = content.split("INSTRUCTION:")[1].split("RESPONSE:")[0].strip()
        answer = content.split("RESPONSE:")[1].strip()
        return instr_part, answer
    except Exception as e:

        return None, None


## Salvar dataset em formato JSONL

In [5]:
def save_to_jsonl(pairs, output_file="dataset.jsonl"):

    with open(output_file, "w", encoding="utf-8") as f:
        for instruction, answer in pairs:
            if instruction and answer:
                example = {
                    "Instruction": instruction,
                    "Output": answer
                }
                f.write(json.dumps(example, ensure_ascii=False) + "\n")

## Gerando o dataset

In [6]:
def generate_dataset(file_path="manual_genie.pdf", model_id="microsoft/Phi-3-mini-4k-instruct", output_file="dataset.jsonl", max_chunks=None):

    print(f"Carregando modelo: {model_id} ...")

    hf_pipeline = pipeline(
        "text-generation",
        model=model_id,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )

    text = extract_text_pdf(file_path)
    chunks = split_text(text)

    if max_chunks:
        chunks = chunks[:max_chunks]

    print(f"Gerando pares (instrução + resposta) para {len(chunks)} chunks...")
    pairs = []

    for chunk in tqdm(chunks, desc="Processando chunks"):
        if len(chunk.strip()) < 10: 
            continue
        instruction, answer = generate_instruction_response(chunk, hf_pipeline)
        if instruction and answer:
            pairs.append((instruction, answer))

    save_to_jsonl(pairs, output_file)
    print(f"\nDataset salvo em: {output_file} ({len(pairs)} exemplos gerados)")


In [7]:
# Gerando dataset
caminho_arquivo = "manual_genie.pdf" 

generate_dataset(
    file_path=caminho_arquivo,
    model_id="microsoft/Phi-4-mini-instruct",
    output_file="dataset_teste.jsonl",
    max_chunks=3
)

Carregando modelo: microsoft/Phi-3-mini-4k-instruct ...


Loading weights: 100%|██████████| 195/195 [00:00<00:00, 2396.54it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


Gerando pares (instrução + resposta) para 3 chunks...


Processando chunks: 100%|██████████| 3/3 [1:09:39<00:00, 1393.24s/it]


Dataset salvo em: dataset_teste.jsonl (0 exemplos gerados)
